# 9.5. Recurrent Neural Network Implementation from Scratch
D2L의 Recurrent Neural Network Implementation from Scratch장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import math
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. RNN 직접 구현

이번 장에서는 PyTorch의 nn.RNN을 사용하지 않고 RNN을 직접 구현한다. 목표는 문자 단위(Character-level) 언어 모델을 만드는 것이다.

예를 들어 모델이 time mach까지 입력받았다면 다음 문자가 무엇일지 예측한다.

전체 구조는 다음과 같다.

```text
문자
↓
One-Hot Encoding
↓
RNN
↓
Hidden State
↓
Fully Connected Layer
↓
다음 문자 예측
```

RNN에서 가장 중요한 것은 현재 입력뿐 아니라 이전 시점의 hidden state도 함께 사용한다는 것이다.

$$
H_t = \tanh(X_tW_{xh} + H_{t-1}W_{hh} + b_h)
$$

## 2. 파라미터

RNN 내부 계산을 직접 만들기 위해 다음 파라미터들을 직접 정의할 것이다.

`W_xh`: 입력 → hidden  
`W_hh`: 이전 hidden → 현재 hidden  
`b_h` : hidden bias

## 3. RNN 파라미터 만들기

In [3]:
class RNNScratch(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()

        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.sigma = sigma

        self.W_xh = nn.Parameter(
            torch.randn(num_inputs, num_hiddens) * sigma
        )

        self.W_hh = nn.Parameter(
            torch.randn(num_hiddens, num_hiddens) * sigma
        )

        self.b_h = nn.Parameter(
            torch.zeros(num_hiddens)
        )

shape를 보면 RNN 구조를 이해하기 쉽다. 현재 입력 X_t는 W_xh와 곱해서 hidden state 크기로 바꾼다.

    X_t (batch_size, num_inputs) @ W_xh (num_inputs, num_hiddens) -> (batch_size, num_hiddens)

이전 hidden state H_(t-1)도 W_hh와 곱한다.

    H_(t-1) (batch_size, num_hiddens) @ W_hh (num_hiddens, num_hiddens) -> (batch_size, num_hiddens)

두 결과의 크기가 같기 때문에 더할 수 있다.

    현재 입력 정보 + 이전 시점의 기억

RNN은 현재 입력과 이전 hidden state를 함께 사용해 새로운 hidden state를 만든다.

## 4. RNN 순전파

RNN은 시간 순서대로 입력을 하나씩 처리한다.

In [4]:
@d2l.add_to_class(RNNScratch)
def forward(self, inputs, state=None):

    if state is None:
        state = torch.zeros(
            (inputs.shape[1], self.num_hiddens),
            device=inputs.device
        )
    else:
        state, = state

    outputs = []

    for X in inputs:

        state = torch.tanh(
            X @ self.W_xh
            + state @ self.W_hh
            + self.b_h
        )

        outputs.append(state)

    return outputs, state

입력 shape = (num_steps, batch_size, num_inputs)

예를 들어 X.shape = (100, 2, 16)이면 for X in inputs를 실행하면 한 번 X는 (2, 16)이다.

한 time step씩 처리한다. 

$$
H_t = \tanh(X_tW_{xh} + H_{t-1}W_{hh} + b_h)
$$

```python
state = torch.tanh(
    X @ self.W_xh
    + state @ self.W_hh
    + self.b_h
)
```

코드로는 여기 부분이다. 현재 X와 이전 기억 state를 합쳐서 새로운 기억을 만든다.

## 5. RNN 출력 Shape 확인

In [5]:
batch_size = 2
num_inputs = 16
num_hiddens = 32
num_steps = 100

rnn = RNNScratch(
    num_inputs=num_inputs,
    num_hiddens=num_hiddens
)

X = torch.ones(
    (num_steps, batch_size, num_inputs)
)

outputs, state = rnn(X)

print(len(outputs))
print(outputs[0].shape)
print(state.shape)

100
torch.Size([2, 32])
torch.Size([2, 32])


100개의 time step 각각에 대해 hidden state가 하나씩 만들어진다.

```text
X1 → H1
X2 + H1 → H2
X3 + H2 → H3
...
X100 + H99 → H100
```

마지막 state는 H100이다.

## 6. RNN을 언어 모델로 만들기

지금까지의 RNN은 hidden state만 만든다. 하지만 언어 모델은 다음 문자가 무엇인지 예측해야 한다.

    Hidden State -> Fully Connected Layer -> Vocabulary 크기 출력

구조가 추가되어야 한다.

$$
O_t = H_tW_{hq} + b_q
$$

여기서 출력 차원은 vocabulary 크기이다. 예를 들어서 사용 가능한 문자가 28개라면 이렇다.

    hidden state(batch, 32) -> output(batch, 28)

In [6]:
class RNNLMScratch(d2l.Classifier):

    def __init__(self, rnn, vocab_size, lr=0.01):
        super().__init__()

        self.rnn = rnn
        self.vocab_size = vocab_size
        self.lr = lr

        self.W_hq = nn.Parameter(
            torch.randn(
                rnn.num_hiddens,
                vocab_size
            ) * rnn.sigma
        )

        self.b_q = nn.Parameter(
            torch.zeros(vocab_size)
        )

## 7. One-Hot Encoding